## Inicializacion

Antes de comenzar, importamos las dependencias necesarias y definimos una funcion auxiliar para interactuar con la Messages API de Anthropic.

In [ ]:
import re
from anthropic import Anthropic

client = Anthropic()

def get_completion(prompt: str, system: str = "", prefill=None, messages=None) -> str:
    
    if messages is None:
        messages = [{"role": "user", "content": prompt}]
    if prefill:
        messages.append({"role": "assistant", "content": prefill})

    kwargs = {
        "model": "claude-haiku-4-5-20251001",
        "max_tokens": 1000,
        "messages": messages,
    }
    if system:
        kwargs["system"] = system

    response = client.messages.create(**kwargs)
    return response.content[0].text


### Qwen3.5

In [30]:
from ollama import chat

def get_completion_ollama(
    prompt: str = "",
    system: str = "",
    prefill: str = None,
    messages: list = None,
    model: str = "qwen3.5:35b-a3b",
    temperature: float = 0.7,
    think: bool = False,
) -> str:
    """
    Wrapper para ollama.chat con defaults razonables.
    
    Si se pasa `messages`, se usa como historial directo.
    Si no, se construye desde `system` y `prompt`.
    """
    # Construir mensajes
    if messages is None:
        messages = []
        if system:
            messages.append({"role": "system", "content": system})
        if prompt:
            messages.append({"role": "user", "content": prompt})
    else:
        # Si se pasó messages, opcionalmente agregar system al inicio
        if system and not any(m["role"] == "system" for m in messages):
            messages = [{"role": "system", "content": system}] + messages
        # Si se pasó prompt además de messages, agregarlo al final como user
        if prompt:
            messages = messages + [{"role": "user", "content": prompt}]
    
    # Prefill opcional
    if prefill:
        messages = messages + [{"role": "assistant", "content": prefill}]
    
    response = chat(
        model=model,
        messages=messages,
        options={
            "temperature": temperature,
            "top_p": 0.8,
            "top_k": 20,
            "repeat_penalty": 1.1,    # ← clave para evitar loops
            "num_predict": 2048,       # ← límite de respuesta
            "num_ctx": 24000,
        },
        think=think,
    )
    
    return response.message.content

## Estructura de una llamada a la Messages API

Una llamada a la Messages API requiere como minimo los siguientes argumentos:

- **model**: el modelo a utilizar (por ejemplo, `claude-haiku-4-5-20251001`)
- **max_tokens**: el numero maximo de tokens que puede contener la respuesta
- **messages**: la lista de mensajes que conforman la conversacion

Adicionalmente, se pueden incluir parametros opcionales como:

- **system**: el system prompt que define el comportamiento general del modelo
- **temperature**: controla la aleatoriedad de las respuestas (0.0 = determinista, 1.0 = mas creativo)

**Reglas importantes sobre `messages`:**

1. Los roles `user` y `assistant` deben alternarse estrictamente. No se pueden tener dos mensajes consecutivos del mismo rol.
2. La conversacion siempre debe comenzar con un mensaje de rol `user`.

Violar cualquiera de estas reglas producira un error en la API.

In [ ]:
SYSTEM = "Eres un asistente experto en historia. Tus respuestas deben ser claras, concisas y bien fundamentadas."
PROMPT = "Explica brevemente las causas principales de la Revolucion Francesa."

print(get_completion(PROMPT, SYSTEM))

### Qwen3.5

In [3]:
SYSTEM = "Eres un asistente experto en historia. Tus respuestas deben ser claras, concisas y bien fundamentadas."
PROMPT = "Explica brevemente las causas principales de la Revolucion Francesa."

print(get_completion_ollama(PROMPT, SYSTEM))

<think>

</think>

La Revolución Francesa (1789) fue el resultado de una convergencia de factores estructurales, económicos e ideológicos que desestabilizaron el Antiguo Régimen:

1.  **Crisis económica y fiscal**: El estado francés estaba en bancarrota debido a los gastos militares (especialmente la ayuda a la Independencia de EE. UU.) y al mantenimiento de la corte. El sistema fiscal era injusto: la nobleza y el clero estaban exentos de impuestos, mientras que la carga recaía casi exclusivamente sobre el Tercer Estado (campesinos, burguesía y trabajadores urbanos).
2.  **Desigualdad social y el sistema de estamentos**: La sociedad estaba rigidamente dividida en tres estados. El Tercer Estado, que constituía el 98% de la población, carecía de poder político a pesar de ser la base económica del reino, lo que generó un profundo resentimiento contra los privilegios de la nobleza y el clero.
3.  **Crisis de subsistencia**: Malas cosechas en 1788 y 1789 provocaron una escasez crítica de tr

### Que es el System Prompt?

El system prompt le indica al modelo como debe comportarse, que tipo de respuestas debe generar y bajo que contexto debe operar. En el ejemplo anterior, le indicamos que actuara como un experto en historia con respuestas claras y fundamentadas.

El system prompt es una herramienta poderosa porque permite moldear el comportamiento del modelo sin necesidad de repetir instrucciones en cada mensaje del usuario. Esto mejora tanto la consistencia como la eficiencia de las respuestas.

## La importancia de dar instrucciones claras

Claude es como una persona que recien se incorpora a un nuevo trabajo: si se le dan instrucciones claras y detalladas, realizara la tarea de manera casi perfecta. Pero si las instrucciones son vagas o ambiguas, el resultado puede no ser el esperado.

Un prompt bien construido marca una gran diferencia en la calidad de la respuesta. Por ejemplo, podemos hacer una pregunta abierta sin system prompt y observar como responde el modelo.

In [ ]:
PROMPT = "Quien es considerado el mejor jugador de ajedrez de todos los tiempos?"
SYSTEM = ""

print(get_completion(PROMPT, SYSTEM))

### QWen3.5

In [5]:
PROMPT = "Quien es considerado el mejor jugador de ajedrez de todos los tiempos?"
SYSTEM = ""

print(get_completion_ollama(PROMPT, SYSTEM))

<think>
Here's a thinking process that leads to the suggested answer:

1.  **Analyze the Request:**
    *   **Question:** "Quien es considerado el mejor jugador de ajedrez de todos los tiempos?" (Who is considered the best chess player of all time?)
    *   **Language:** Spanish.
    *   **Subject:** Chess history/players.
    *   **Nature of the question:** Subjective but has a widely accepted consensus, though there are nuances (different eras, different metrics).

2.  **Identify the Consensus Candidate:**
    *   When people ask this, the name that almost always comes up first is **Garry Kasparov**.
    *   However, **Magnus Carlsen** is the current reigning king and has the highest rating ever.
    *   **Bobby Fischer** is often cited for his peak dominance (1972).
    *   **Emanuel Lasker** held the title for the longest time.
    *   **Jose Raul Capablanca** is revered for his natural talent.
    *   **Anatoly Karpov** is often compared to Kasparov.
    *   **Paul Morphy** is the

Se puede observar que el modelo ofrece varias opciones o matiza su respuesta. Si necesitamos una respuesta unica y definitiva, debemos ser mas especificos en nuestra instruccion. La claridad y precision en el prompt es clave para obtener el tipo de respuesta que buscamos.

In [ ]:
PROMPT = "Quien es considerado el mejor jugador de ajedrez de todos los tiempos? Elige solo uno y justifica tu eleccion brevemente."
SYSTEM = ""

print(get_completion(PROMPT, SYSTEM))

### Qwen 3.5

In [6]:
PROMPT = "Quien es considerado el mejor jugador de ajedrez de todos los tiempos? Elige solo uno y justifica tu eleccion brevemente."
SYSTEM = ""

print(get_completion_ollama(PROMPT, SYSTEM))

<think>

</think>

**Garry Kasparov** es ampliamente considerado el mejor jugador de ajedrez de todos los tiempos.

Su elección se justifica principalmente por su dominio absoluto durante más de dos décadas, habiendo mantenido el número uno mundial durante **255 meses consecutivos** (el récord histórico) y siendo el primer jugador en superar la barrera de los **2800 puntos ELO**. Kasparov combinó una profundidad estratégica inigualable con un juego táctico feroz, dominando la era clásica y venciendo a sus rivales más grandes, incluidos Anatoly Karpov y el propio Magnus Carlsen en enfrentamientos tempranos, antes de que la inteligencia artificial cambiara el juego.


## Prompt Templates

Una tecnica muy util es el uso de **prompt templates**: plantillas de prompts donde solo se cambian pequenas partes (variables) para reutilizar la misma estructura con diferentes datos de entrada. Esto es especialmente valioso cuando se necesita ejecutar la misma tarea sobre multiples conjuntos de datos.

En Python, se pueden usar f-strings o el metodo `.format()` para insertar variables dinamicamente en el prompt.

In [7]:
DATASET_1 = [
    {"nombre": "Alice", "edad": 30, "ciudad": "Madrid"},
    {"nombre": "Bob", "edad": 25, "ciudad": "Barcelona"},
    {"nombre": "Charlie", "edad": 35, "ciudad": "Buenos Aires"}
]

DATASET_2 = [
    {"producto": "Laptop", "precio": 999.99, "stock": 10},
    {"producto": "Smartphone", "precio": 499.99, "stock": 20},
    {"producto": "Tablet", "precio": 299.99, "stock": 15}
]

SYSTEM = "Eres un experto en analisis de datos. Tus respuestas deben ser claras y detalladas (al menos 200 palabras)."
PROMPT_TEMPLATE = "Por favor analiza el siguiente conjunto de datos y proporciona un resumen detallado: {datos}"

In [ ]:


print("--- Analisis Dataset 1 ---")
print(get_completion(PROMPT_TEMPLATE.format(datos=str(DATASET_1)), SYSTEM))

print("\n--- Analisis Dataset 2 ---")
print(get_completion(PROMPT_TEMPLATE.format(datos=str(DATASET_2)), SYSTEM))

### Qwen 3.5

In [8]:
print("--- Analisis Dataset 1 ---")
print(get_completion_ollama(PROMPT_TEMPLATE.format(datos=str(DATASET_1)), SYSTEM))

print("\n--- Analisis Dataset 2 ---")
print(get_completion_ollama(PROMPT_TEMPLATE.format(datos=str(DATASET_2)), SYSTEM))

--- Analisis Dataset 1 ---
<think>

</think>

El conjunto de datos proporcionado consta de un registro inicial que incluye a tres individuos: Alice, Bob y Charlie. Cada entrada contiene tres variables clave: el nombre del sujeto, su edad y su ciudad de residencia. Al realizar un análisis descriptivo detallado, observamos que la muestra es pequeña pero ofrece una visión clara de la diversidad geográfica y demográfica presente. En cuanto a la variable numérica "edad", los valores oscilan entre 25 y 35 años. Específicamente, Bob es el miembro más joven del grupo con 25 años, mientras que Charlie es el mayor con 35 años. Alice se sitúa en un punto intermedio con 30 años. Si calculamos la media aritmética de las edades, obtenemos un promedio de 30 años, lo que sugiere que el grupo se compone de adultos jóvenes con una distribución bastante equilibrada alrededor de este valor central.

Desde una perspectiva cualitativa, la variable "ciudad" revela una distribución internacional interesante. 

## Etiquetas XML para estructurar prompts

Claude fue entrenado con datos que incluyen etiquetas XML, por lo que las comprende y utiliza de manera natural. Encerrar datos dentro de etiquetas XML como `<datos></datos>` ofrece varias ventajas:

1. **Separacion clara** entre instrucciones y datos, lo que reduce la ambiguedad
2. **Mejor comprension** del prompt por parte del modelo, especialmente cuando los datos son complejos o contienen ruido
3. **Respuestas estructuradas**: se le puede pedir al modelo que responda dentro de etiquetas XML especificas, lo que facilita la extraccion programatica de la informacion relevante

In [15]:
SYSTEM = "Eres un experto en analisis de datos. Tus respuestas deben ser claras y detalladas (al menos 200 palabras)."

# Nota: no se puede usar .format() directamente porque str() de los dicts contiene { } 
# que .format() interpreta como placeholders. Usamos f-string en su lugar.
def make_prompt(datos):
    return f"Por favor analiza el siguiente conjunto de datos y proporciona un resumen detallado: <datos>{datos}</datos>"

In [ ]:

print("--- Analisis Dataset 1 ---")
print(get_completion(make_prompt(DATASET_1), SYSTEM))

print("\n--- Analisis Dataset 2 ---")
print(get_completion(make_prompt(DATASET_2), SYSTEM))

### Qwen 3.5

In [16]:
print("--- Analisis Dataset 1 ---")
print(get_completion_ollama(make_prompt(DATASET_1), SYSTEM))

--- Analisis Dataset 1 ---
<think>

</think>

El conjunto de datos proporcionado consta de tres registros individuales que representan a personas con información demográfica básica, específicamente su nombre, edad y ciudad de residencia. Al realizar un análisis exploratorio inicial, observamos que la muestra es extremadamente reducida, lo que limita la capacidad de extraer conclusiones estadísticas robustas o tendencias generales sobre poblaciones más amplias. Sin embargo, es posible desglosar las características presentes en cada fila para entender la estructura y el contenido de la información.

En cuanto a la variable "edad", los valores registrados son 30, 25 y 35 años. Esto indica un rango de edad de 10 años, con una edad mínima de 25 años (Bob) y una máxima de 35 años (Charlie). El cálculo de la edad promedio para este grupo específico resulta ser de 30 años, lo que sugiere que los individuos se encuentran en etapas tempranas de la vida adulta. Respecto a la ubicación geográfica,

In [17]:
print("\n--- Analisis Dataset 2 ---")
print(get_completion_ollama(make_prompt(DATASET_2), SYSTEM))


--- Analisis Dataset 2 ---
<think>
Thinking Process:

1.  **Analyze the Request:**
    *   **Role:** Data Analysis Expert.
    *   **Task:** Analyze the provided dataset and provide a detailed summary.
    *   **Constraint:** Answers must be clear and detailed (at least 200 words).
    *   **Input Data:** A list of dictionaries containing product information (product name, price, stock).
        *   Laptop: 999.99, 10 units.
        *   Smartphone: 499.99, 20 units.
        *   Tablet: 299.99, 15 units.

2.  **Analyze the Data:**
    *   **Structure:** JSON-like list of objects.
    *   **Fields:** 'producto' (categorical), 'precio' (numerical), 'stock' (numerical).
    *   **Items:** 3 products.
    *   **Calculations:**
        *   Total Inventory Value (Price * Stock):
            *   Laptop: 999.99 * 10 = 9999.90
            *   Smartphone: 499.99 * 20 = 9999.80
            *   Tablet: 299.99 * 15 = 4499.85
            *   Total Value: 9999.90 + 9999.80 + 4499.85 = 24499.55
      

### Prefilling: poner palabras en la boca de Claude

Ademas de usar etiquetas XML en la entrada, se puede forzar al modelo a responder dentro de etiquetas XML especificas mediante la tecnica de **prefilling**. Esta tecnica consiste en iniciar la respuesta del asistente con un texto predefinido (por ejemplo, la etiqueta de apertura `<resumen>`), de modo que el modelo continue generando desde ese punto.

Esto es util para:

- Garantizar un formato de salida consistente y predecible
- Facilitar el parseo automatico de las respuestas
- Guiar al modelo hacia el tipo de contenido deseado

In [21]:
SYSTEM = "Eres un experto en analisis de datos. Tus respuestas deben ser claras y detalladas (al menos 200 palabras)."
PREFILL = "<resumen>"

def make_prompt_resumen(datos):
    return f"Analiza el siguiente conjunto de datos y proporciona un resumen detallado dentro de etiquetas <resumen>: <datos>{datos}</datos>"



In [ ]:
print("--- Analisis Dataset 1 ---")
print(get_completion(make_prompt_resumen(DATASET_1), SYSTEM, PREFILL))

print("\n--- Analisis Dataset 2 ---")
print(get_completion(make_prompt_resumen(DATASET_2), SYSTEM, PREFILL))

### Qwen3.5

In [22]:
# Prefilling no funciona bien en modelos open-source. 
# En su lugar, pedimos el formato directamente en el prompt.
def make_prompt_resumen_ollama(datos):
    return f"Analiza el siguiente conjunto de datos y proporciona un resumen detallado. Responde UNICAMENTE dentro de etiquetas <resumen></resumen>: <datos>{datos}</datos>"

print("--- Analisis Dataset 1 ---")
print(get_completion_ollama(make_prompt_resumen_ollama(DATASET_1), SYSTEM))

print("\n--- Analisis Dataset 2 ---")
print(get_completion_ollama(make_prompt_resumen_ollama(DATASET_2), SYSTEM))

--- Analisis Dataset 1 ---
<think>
Thinking Process:

1.  **Analyze the Request:**
    *   **Role:** Data Analysis Expert.
    *   **Task:** Analyze the provided dataset and provide a detailed summary.
    *   **Constraint 1:** Answers must be clear and detailed (at least 200 words).
    *   **Constraint 2:** Answer MUST be ONLY within `<resumen></resumen>` tags.
    *   **Input Data:** A list of dictionaries containing 'nombre' (name), 'edad' (age), and 'ciudad' (city) for three individuals: Alice, Bob, and Charlie.

2.  **Analyze the Data:**
    *   **Records:** 3 entries.
    *   **Fields:**
        *   `nombre`: Categorical (Alice, Bob, Charlie).
        *   `edad`: Numerical (30, 25, 35).
        *   `ciudad`: Categorical (Madrid, Barcelona, Buenos Aires).
    *   **Observations:**
        *   Ages range from 25 to 35.
        *   Average age calculation: (30 + 25 + 35) / 3 = 90 / 3 = 30.
        *   Cities are in different countries (Spain, Argentina).
        *   No duplicates i

## Chain of Thought (Pensamiento paso a paso)

Incluso con un buen system prompt y contexto adecuado, el modelo puede cometer errores en tareas complejas que requieren razonamiento en multiples pasos. Una tecnica efectiva para mejorar la calidad de las respuestas es el **chain of thought**: pedirle al modelo que razone paso a paso antes de dar su respuesta final.

Es equivalente a darle a una persona no solo la tarea, sino tambien un procedimiento detallado de como abordarla. Al descomponer el razonamiento en pasos explicitos, el modelo tiende a producir respuestas mas precisas y fundamentadas.

In [ ]:
SYSTEM = """Eres un experto en analisis de datos. Tus respuestas deben ser claras y detalladas (al menos 200 palabras).
Antes de responder, sigue estos pasos:
1. Analiza el numero de entradas y sus atributos
2. Identifica patrones o correlaciones entre los atributos
3. Genera un resumen detallado de los hallazgos
Responde dentro de etiquetas <resumen>."""

PREFILL = "<resumen>"

PROMPT = f"Analiza el siguiente conjunto de datos paso a paso: <datos>{str(DATASET_1)}</datos>"
print("--- Analisis Dataset 1 ---")
print(get_completion(PROMPT, SYSTEM, PREFILL))

PROMPT = f"Analiza el siguiente conjunto de datos paso a paso: <datos>{str(DATASET_2)}</datos>"
print("\n--- Analisis Dataset 2 ---")
print(get_completion(PROMPT, SYSTEM, PREFILL))

### Qwen3.5

In [23]:
SYSTEM_COT = """Eres un experto en analisis de datos. Tus respuestas deben ser claras y detalladas (al menos 200 palabras).
Antes de responder, sigue estos pasos:
1. Analiza el numero de entradas y sus atributos
2. Identifica patrones o correlaciones entre los atributos
3. Genera un resumen detallado de los hallazgos
Responde dentro de etiquetas <resumen>."""

def make_prompt_cot(datos):
    return f"Analiza el siguiente conjunto de datos paso a paso: <datos>{datos}</datos>"

print("--- Analisis Dataset 1 ---")
print(get_completion_ollama(make_prompt_cot(DATASET_1), SYSTEM_COT))

print("\n--- Analisis Dataset 2 ---")
print(get_completion_ollama(make_prompt_cot(DATASET_2), SYSTEM_COT))

--- Analisis Dataset 1 ---
<think>

</think>

<resumen>
El conjunto de datos proporcionado consta de tres registros individuales, cada uno representando a una persona con atributos específicos: nombre, edad y ciudad de residencia. Al analizar la estructura, observamos que el dataset es pequeño pero contiene información demográfica básica que permite realizar una descripción inicial. En primer lugar, los nombres son únicos para cada entrada (Alice, Bob y Charlie), lo que indica que no hay duplicados en la identificación de los sujetos. En segundo lugar, al examinar la variable numérica "edad", encontramos un rango que va desde los 25 años (Bob) hasta los 35 años (Charlie), con un promedio de 30 años. Esto sugiere que el grupo está compuesto por adultos jóvenes en su etapa laboral temprana o media.

Respecto a la variable geográfica "ciudad", los datos muestran una distribución internacional, abarcando dos países distintos: España (Madrid y Barcelona) y Argentina (Buenos Aires). No exist

## Few-Shot Prompting

El **few-shot prompting** consiste en proporcionarle al modelo uno o mas ejemplos del comportamiento deseado antes de hacerle la solicitud real. Estos ejemplos pueden incluirse de dos formas: directamente en el texto del prompt, o como mensajes previos en la secuencia de conversacion (alternando roles `user` y `assistant`).

La nomenclatura es la siguiente:

- **Zero-shot**: sin ejemplos previos, solo la instruccion
- **One-shot**: un unico ejemplo antes de la solicitud
- **Few-shot**: varios ejemplos
- **N-shot**: n ejemplos en general

Cuantos mas ejemplos relevantes y representativos se proporcionen, mejor entendera el modelo el patron esperado.

In [ ]:
PROMPT = """Por favor completa la conversacion escribiendo la siguiente linea. Tu eres el Asistente:

<conversacion>
Usuario: Buenos dias, como se encuentra?
Asistente: Muy bien, gracias por preguntar. En que puedo ayudarle hoy?
Usuario: Me gustaria saber sobre cursos de programacion disponibles.
</conversacion>
"""

print(get_completion(PROMPT))

# Nota: lo anterior es equivalente a usar la secuencia de mensajes directamente:
# messages = [
#     {"role": "user", "content": "Buenos dias, como se encuentra?"},
#     {"role": "assistant", "content": "Muy bien, gracias por preguntar. En que puedo ayudarle hoy?"},
#     {"role": "user", "content": "Me gustaria saber sobre cursos de programacion disponibles."}
# ]

### Qwen3.5

In [24]:
PROMPT = """Por favor completa la conversacion escribiendo la siguiente linea. Tu eres el Asistente:

<conversacion>
Usuario: Buenos dias, como se encuentra?
Asistente: Muy bien, gracias por preguntar. En que puedo ayudarle hoy?
Usuario: Me gustaria saber sobre cursos de programacion disponibles.
</conversacion>
"""

print(get_completion_ollama(PROMPT))

<think>

</think>

¡Excelente! Tenemos varios cursos de programación disponibles para diferentes niveles y lenguajes. ¿Te interesa alguna tecnología en particular, como Python, JavaScript, Java o C++, o prefieres que te cuente un poco sobre nuestras opciones más populares?


## Reducir alucinaciones

En ocasiones, el modelo puede generar informacion incorrecta o inventada. Esto no se debe a un fallo, sino a su tendencia a intentar ser util incluso cuando no tiene informacion suficiente para responder con precision. A este fenomeno se le conoce como **alucinacion** (hallucination).

Para mitigar este problema, existen varias estrategias:

- **Dar una salida explicita**: indicarle al modelo que si no sabe algo, responda honestamente que no lo sabe en lugar de inventar una respuesta
- **Reducir la temperature**: valores mas bajos (cercanos a 0) producen respuestas mas conservadoras y menos propensas a la invencion
- **Pedir evidencia o justificacion**: solicitar que cite fuentes o fundamente sus afirmaciones

In [ ]:
SYSTEM = "Eres un asistente de conocimiento general. Responde con precision y honestidad."
PROMPT = "Cual es la poblacion exacta de la ciudad de Atlantida? Solo responde si sabes la respuesta con certeza. De lo contrario, indica que no tienes esa informacion."

print(get_completion(PROMPT, SYSTEM))

### Qwen3.5

In [25]:
SYSTEM = "Eres un asistente de conocimiento general. Responde con precision y honestidad."
PROMPT = "Cual es la poblacion exacta de la ciudad de Atlantida? Solo responde si sabes la respuesta con certeza. De lo contrario, indica que no tienes esa informacion."

print(get_completion_ollama(PROMPT, SYSTEM))

<think>

</think>

No tengo esa información con certeza.

La población de la ciudad de Atlántida (comúnmente referida como **Atlántida, Uruguay**, o **Atlántida, Argentina**) varía constantemente y los datos exactos dependen del censo más reciente y la fuente oficial (como el INE en Uruguay o el INDEC en Argentina). Dado que no tengo acceso a datos censales en tiempo real ni a cifras oficiales actualizadas al último día, no puedo proporcionar un número exacto sin riesgo de inexactitud.

Si necesitas una cifra precisa, te recomiendo consultar el último censo nacional de tu país o los informes demográficos oficiales de la municipalidad correspondiente.


## Guia de estructura para prompts complejos

Cuando se trabaja con tareas complejas, es recomendable seguir una estructura organizada para construir el prompt. A continuacion se presenta una guia con los elementos clave, en orden sugerido.

---

### Elemento 1: Rol `user`

La llamada a la Messages API siempre debe comenzar con un mensaje de rol `user` en el array de mensajes.

### Elemento 2: Contexto de la tarea

Proporcionar al modelo el contexto sobre el rol que debe asumir, los objetivos generales y la tarea a realizar. Es recomendable colocar el contexto al inicio del prompt.

**Ejemplo:**
```
Actuaras como un asistente de atencion al cliente creado por la empresa X. Tu objetivo es resolver dudas de los usuarios que visitan el sitio web de la empresa.
```

### Elemento 3: Contexto de tono

Si es relevante para la interaccion, indicar al modelo que tono debe utilizar. Este elemento no siempre es necesario, depende de la tarea.

**Ejemplo:**
```
Debes mantener un tono profesional y amigable en todo momento.
```

### Elemento 4: Descripcion detallada de la tarea y reglas

Expandir las tareas especificas que el modelo debe realizar, asi como las reglas que debe seguir. Aqui tambien se le puede dar una "salida" para cuando no tenga la respuesta.

**Ejemplo:**
```
Reglas importantes para la interaccion:
- Mantente siempre en el rol asignado
- Si no estas seguro de como responder, di: "Disculpa, no entendi tu pregunta. Podrias reformularla?"
- Si alguien pregunta algo fuera de tema, redirige amablemente hacia el proposito de la conversacion
```

### Elemento 5: Ejemplos (few-shot)

Proporcionar al modelo al menos un ejemplo de la respuesta ideal que se espera. Encerrar los ejemplos en etiquetas XML. Si se proporcionan multiples ejemplos, cada uno debe estar en su propio par de etiquetas.

Los ejemplos son probablemente la herramienta mas efectiva para lograr que el modelo se comporte como se desea. Se recomienda incluir ejemplos de casos comunes y casos limite.

**Ejemplo:**
```
Aqui hay un ejemplo de como responder en una interaccion estandar:

<ejemplo>
Usuario: Hola, que haces y como fuiste creado?
Asistente: Hola! Soy un asistente virtual creado por la empresa X para ayudarte con tus consultas. En que puedo asistirte hoy?
</ejemplo>
```

### Elemento 6: Datos de entrada

Si el modelo necesita procesar datos especificos, incluirlos dentro de etiquetas XML relevantes. Se pueden incluir multiples bloques de datos, cada uno en sus propias etiquetas.

**Ejemplo:**
```
Aqui esta el historial de la conversacion previa entre el usuario y tu:
<historial>
{historial_conversacion}
</historial>

Aqui esta la pregunta actual del usuario:
<pregunta>
{pregunta_usuario}
</pregunta>
```

### Elemento 7: Solicitud inmediata

"Recordarle" al modelo exactamente que debe hacer para cumplir con la tarea. Es mejor colocar esto hacia el final del prompt, ya que produce mejores resultados que ponerlo al principio. Tambien es buena practica colocar la consulta del usuario cerca del final.

**Ejemplo:**
```
Como respondes a la pregunta del usuario?
```

### Elemento 8: Precognicion (pensamiento paso a paso)

Para tareas con multiples pasos, indicar al modelo que piense paso a paso antes de dar una respuesta. En ocasiones es necesario usar frases como "Antes de dar tu respuesta..." para asegurar que el modelo realice el razonamiento previo.

No es necesario en todos los prompts, pero cuando se incluye, es mejor colocarlo hacia el final y justo despues de la solicitud inmediata.

**Ejemplo:**
```
Piensa en tu respuesta paso a paso antes de responder.
```

### Elemento 9: Formato de salida

Si se requiere un formato especifico en la respuesta, indicarlo claramente. Es mejor colocar esto hacia el final del prompt.

**Ejemplo:**
```
Coloca tu respuesta dentro de etiquetas <respuesta>.
```

### Elemento 10: Prefilling de la respuesta

Un espacio para iniciar la respuesta del modelo con texto predefinido que guie su comportamiento. Si se desea usar prefilling, debe colocarse en el rol `assistant` de la llamada a la API.

**Ejemplo:**
```
[Asistente]
```

---

### Resumen

No todos los elementos son necesarios para cada prompt. La estructura y los elementos a incluir dependen de la complejidad de la tarea. Sin embargo, para tareas complejas, seguir esta guia ayuda a obtener respuestas mas consistentes y de mayor calidad.

## Prompt Chaining: encadenar llamadas

Existen varios patrones que permiten mejorar sustancialmente la calidad de las respuestas del modelo. La idea fundamental es utilizar la salida de una llamada como entrada de la siguiente, creando una cadena de procesamiento. Los tres patrones mas comunes son:

1. **Revision**: generar una respuesta y luego pedirle al modelo que la revise y corrija
2. **Mejora iterativa**: generar un borrador y luego pedir que lo refine con instrucciones especificas
3. **Pipeline de procesamiento**: usar el resultado de un prompt como dato de entrada para otro prompt con una tarea completamente diferente

### Patron 1: Revision

El patron de revision consiste en pedirle al modelo que genere una respuesta y luego, en un segundo turno, que la revise y corrija. Esto es especialmente util cuando la tarea inicial es propensa a errores o alucinaciones, ya que el modelo puede detectar y corregir sus propios fallos al examinar su salida con ojos criticos.

In [ ]:
# Paso 1: El modelo genera una respuesta inicial
first_response = get_completion(
    "Genera una lista de 10 palabras poco comunes en espanol y su significado."
)
print("--- Respuesta inicial ---")
print(first_response)

# Paso 2: Pedirle que revise su propia respuesta
messages = [
    {"role": "user", "content": "Genera una lista de 10 palabras poco comunes en espanol y su significado."},
    {"role": "assistant", "content": first_response},
    {"role": "user", "content": "Revisa la lista anterior. Verifica que todas las palabras existan realmente y que los significados sean correctos. Reemplaza cualquier palabra incorrecta o inventada."}
]

revision = get_completion("", messages=messages)
print("\n--- Respuesta revisada ---")
print(revision)

### Qwen3.5

In [33]:
# Paso 1: respuesta inicial
first_response = get_completion_ollama(
    prompt="Give me 10 english rare words.",
    model='gemma4-26b-heretic:latest',
)
print("--- Respuesta inicial ---")
print(first_response)
print(f"\n[Tokens aprox: {len(first_response) // 4}]")

# Paso 2: revisión
messages = [
    {"role": "user", "content": "Generate a list of 10 rare English words and their meanings."},
    {"role": "assistant", "content": first_response},
    {"role": "user", "content": "Review the previous list. Verify that all the words actually exist and that the meanings are correct. Replace any incorrect or made-up words."}
]

revision = get_completion_ollama(messages=messages)
print("\n--- Respuesta revisada ---")
print(revision)

--- Respuesta inicial ---
Here are 10 rare and beautiful English words, along with their definitions and examples of how to use them:

1. **Petrichor** (noun)
   *   **Definition:** The pleasant, earthy smell that accompanies the first rain after a long period of warm, dry weather.
   *   *Example:* "As the storm broke, the air was filled with the soothing scent of **petrichor**."

2. **Ephemeral** (adjective)
   *   **Definition:** Lasting for a very short time; transitory.
   *   *Example:* "The beauty of a sunset is **ephemeral**, vanishing into darkness within minutes."

3. **Mellifluous** (adjective)
   *   **Definition:** (Of a voice or words) sweet or musical; pleasant to hear.
   *   *Example:* "The singer captivated the audience with her **mellifluous** soprano voice."

4. **Ethereal** (adjective)
   *   **Definition:** Extremely delicate and light in a way that seems too perfect for this world; heavenly.
   *   *Example:* "The morning mist gave the lake an **ethereal** appear

El modelo revisa su propia respuesta y corrige los errores que detecta. Sin embargo, existe un problema comun: si la respuesta original ya era correcta, el modelo a veces la modifica innecesariamente porque interpreta que se le esta pidiendo que cambie algo.

La solucion es **darle una salida explicita**: indicarle que si todo esta correcto, lo deje tal cual. De esta forma, el modelo solo modifica lo que realmente necesita correccion.

In [ ]:
# Instruccion de revision mejorada con salida explicita
revision_prompt = (
    "Revisa las palabras anteriores. Reemplaza las que no sean reales. "
    "Si TODAS son correctas, mantenlas tal cual -- no cambies palabras correctas."
)

messages = [
    {"role": "user", "content": "Genera una lista de 10 palabras poco comunes en espanol y su significado."},
    {"role": "assistant", "content": first_response},
    {"role": "user", "content": revision_prompt}
]

revision_mejorada = get_completion("", messages=messages)
print(revision_mejorada)

### Qwen3.5

In [35]:
# Instruccion de revision mejorada con salida explicita
revision_prompt = (
    "Revisa las palabras anteriores. Reemplaza las que no sean reales. "
    "Si TODAS son correctas, mantenlas tal cual -- no cambies palabras correctas."
)

messages = [
    {"role": "user", "content": "Genera una lista de 10 palabras poco comunes en espanol y su significado."},
    {"role": "assistant", "content": first_response},
    {"role": "user", "content": revision_prompt}
]

revision_mejorada_ollama = get_completion_ollama("", messages=messages, model='gemma4-26b-heretic:latest')
print(revision_mejorada_ollama)

Al revisar la lista anterior, noto que cometí un error de contexto: **te proporcioné la lista en inglés cuando me pediste palabras en español.** Todas las palabras que listé existen y son correctas en inglés, pero para cumplir estrictamente con tu petición de "palabras en español", aquí tienes la lista corregida y adaptada al idioma solicitado:

Aquí tienes 10 palabras poco comunes (o cultas) en **español** y su significado:

1. **Petricor**: El nombre que recibe el olor a lluvia sobre suelos secos.
2. **Efímero**: Aquello que dura por muy poco tiempo; pasajero.
3. **Melifluo**: Un sonido o una voz que es excesivamente dulce, suave o delicada.
4. **Etéreo**: Algo extremadamente delicado, ligero, algo que parece fuera de este mundo o celestial.
5. **Limerencia**: El estado mental de estar obsesionado o profundamente enamorado de otra persona.
6. **Serendipia**: Un hallazgo afortunado e inesperado que se produce cuando se está buscando otra cosa distinta.
7. **Inefable**: Algo tan increí

### Patron 2: Mejora iterativa

Este patron consiste en generar un borrador inicial y luego pedirle al modelo que lo mejore con instrucciones especificas. La diferencia con la revision es que aqui no se busca corregir errores, sino elevar la calidad general del contenido. Es equivalente a un proceso de edicion real, donde un primer borrador se refina sucesivamente.

In [ ]:
# Paso 1: Generar un borrador inicial
borrador = get_completion("Escribe un parrafo corto explicando que es la inteligencia artificial.")
print("--- Borrador v1 ---")
print(borrador)

# Paso 2: Pedir una mejora especifica
messages = [
    {"role": "user", "content": "Escribe un parrafo corto explicando que es la inteligencia artificial."},
    {"role": "assistant", "content": borrador},
    {"role": "user", "content": "Mejora este texto. Agrega ejemplos concretos, usa un lenguaje mas preciso y haz que la conclusion sea mas impactante."}
]

version_mejorada = get_completion("", messages=messages)
print("\n--- Borrador v2 (mejorado) ---")
print(version_mejorada)

### Qwen3.5

In [36]:
# Paso 1: Generar un borrador inicial
borrador_ollama = get_completion_ollama("Write a short paragraph explaining what artificial intelligence is.", model='gemma4-26b-heretic:latest')
print("--- Borrador v1 ---")
print(borrador_ollama)

# Paso 2: Pedir una mejora especifica
messages = [
    {"role": "user", "content": "Write a short paragraph explaining what artificial intelligence is."},
    {"role": "assistant", "content": borrador_ollama},
    {"role": "user", "content": "Improve this text. Add concrete examples, use more precise language, and make the conclusion more impactful."}
]

version_mejorada_ollama = get_completion_ollama("", messages=messages, model='gemma4-26b-heretic:latest')
print("\n--- Borrador v2 (mejorado) ---")
print(version_mejorada_ollama)

--- Borrador v1 ---
Artificial intelligence (AI) is a field of computer science dedicated to creating systems capable of performing tasks that typically require human intelligence. Rather than simply following a rigid set of pre-programmed instructions, AI uses algorithms and vast amounts of data to recognize patterns, learn from experience, and make decisions. This allows machines to perform complex functions such as understanding natural language, recognizing images, solving problems, and predicting future outcomes. Ultimately, the goal of AI is to simulate cognitive processes, enabling technology to adapt to new information and execute tasks with increasing efficiency and autonomy.

--- Borrador v2 (mejorado) ---
Artificial intelligence (AI) is a branch of computer science focused on developing systems capable of executing cognitive functions traditionally associated with human intellect, such as reasoning, learning, and problem-solving. Unlike conventional software that relies on s

La segunda version suele ser notablemente mejor porque el modelo puede analizar lo que genero previamente y refinarlo con instrucciones especificas. La clave esta en dar directrices concretas sobre que aspectos mejorar (ejemplos, precision, estructura, etc.) en lugar de pedir simplemente "hazlo mejor".

### Patron 3: Pipeline de procesamiento

En este patron, el resultado de un prompt se utiliza como dato de entrada para otro prompt con una tarea completamente diferente. A diferencia de la revision o la mejora iterativa, aqui cada paso realiza una tarea distinta, formando un pipeline donde la informacion fluye de una etapa a la siguiente.

Esto es util para descomponer tareas complejas en pasos simples y especializados, donde cada paso se puede optimizar de forma independiente.

In [ ]:
texto_largo = """La doctora Maria Lopez presento su investigacion sobre cambio climatico 
en la conferencia de Berlin. El profesor Carlos Ruiz, de la Universidad de Buenos Aires, 
colaboro en el estudio junto con la ingeniera Ana Torres del MIT."""

# Paso 1: Extraer nombres propios del texto
PROMPT_EXTRACCION = f"""Extrae todos los nombres propios de personas del siguiente texto.
Devuelvelos como una lista separada por comas.
<texto>{texto_largo}</texto>"""

nombres = get_completion(PROMPT_EXTRACCION)
print("--- Nombres extraidos ---")
print(nombres)

# Paso 2: Usar los nombres extraidos como entrada para otra tarea
PROMPT_BIOGRAFIAS = f"""Escribe una biografia ficticia de un parrafo para cada una de las 
siguientes personas: {nombres}"""

biografias = get_completion(PROMPT_BIOGRAFIAS)
print("\n--- Biografias generadas ---")
print(biografias)

### Qwen3.5

In [38]:
texto_largo = """La doctora Maria Lopez presento su investigacion sobre cambio climatico 
en la conferencia de Berlin. El profesor Carlos Ruiz, de la Universidad de Buenos Aires, 
colaboro en el estudio junto con la ingeniera Ana Torres del MIT."""

# Paso 1: Extraer nombres propios del texto
PROMPT_EXTRACCION = f"""Extract all proper names of people from the following text.
Return them as a comma-separated list.
<text>{texto_largo}</text>"""

nombres_ollama = get_completion_ollama(PROMPT_EXTRACCION, model='gemma4-26b-heretic:latest')
print("--- Nombres extraidos ---")
print(nombres_ollama)

# Paso 2: Usar los nombres extraidos como entrada para otra tarea
PROMPT_BIOGRAFIAS = f"""Write a fictional one-paragraph biography for each of the 
following people: {nombres_ollama}"""

biografias_ollama = get_completion_ollama(PROMPT_BIOGRAFIAS, model='gemma4-26b-heretic:latest')
print("\n--- Biografias generadas ---")
print(biografias_ollama)

--- Nombres extraidos ---
Maria Lopez, Carlos Ruiz, Ana Torres

--- Biografias generadas ---
**Maria Lopez**
A renowned marine biologist with a penchant for mystery, Maria Lopez spent her youth navigating the rugged coastlines of the Mediterranean. After earning her doctorate in oceanography, she dedicated her life to studying the bioluminescent organisms of the deep Atlantic, a pursuit that earned her both scientific acclaim and a reputation for being a bit of an eccentric hermit. When she isn't documenting rare jellyfish species in her high-tech submersible, Maria can be found in her small seaside cottage, translating ancient maritime poetry and tending to a garden of salt-resistant succulents.

**Carlos Ruiz**
Carlos Ruiz is a master clockmaker whose workshop in the heart of Madrid serves as a sanctuary for lost time. Known for his uncanny ability to repair mechanisms thought to be beyond saving, Carlos possesses a patience that borders on the supernatural. Though he lives a quiet l

## Tool Use (Function Calling)

El uso de herramientas (tool use) permite que el modelo interactue con funciones externas. Es importante entender que **el modelo no ejecuta codigo directamente**. Lo que hace es generar una solicitud estructurada indicando que funcion quiere llamar y con que parametros. El desarrollador ejecuta esa funcion en su propio entorno y le devuelve el resultado al modelo para que continue la conversacion.

En esencia, tool use es una combinacion de dos tecnicas ya vistas:

- **Sustitucion**: insertar resultados de funciones dentro de prompts
- **Prompt chaining**: encadenar el resultado de vuelta al modelo

El proceso se divide en varios pasos que se describen a continuacion.

### Paso 1: Definir las herramientas en el System Prompt

El system prompt para tool use tiene dos partes: una seccion general que explica el mecanismo de llamada, y una seccion especifica que define las herramientas disponibles.

**Parte 1 -- Explicacion general (reutilizable para cualquier proyecto):**

Esta seccion le explica al modelo el mecanismo de llamada a herramientas: como debe formatear sus solicitudes cuando necesite usar una funcion.

In [ ]:
system_tools_general = """Tienes acceso a un conjunto de herramientas que puedes usar 
para responder la pregunta del usuario.

Cuando necesites usar una herramienta, escribe la llamada dentro de 
etiquetas <function_calls> con el siguiente formato:

<function_calls>
    <invoke name="nombre_herramienta">
        <parameter name="param1">valor1</parameter>
        <parameter name="param2">valor2</parameter>
    </invoke>
</function_calls>
"""

**Parte 2 -- Herramientas especificas (cambia segun el caso de uso):**

Aqui se definen las herramientas concretas disponibles, con su nombre, descripcion y parametros. La descripcion de cada herramienta es fundamental, ya que es lo que el modelo lee para decidir cuando usarla y cuando no.

In [ ]:
system_tools_specific = """
<tool_description>
    <tool_name>calculadora</tool_name>
    <description>
        Realiza operaciones aritmeticas basicas.
        Usar cuando se necesite calcular algo.
    </description>
    <parameters>
        <parameter>
            <name>primer_operando</name>
            <type>int</type>
            <description>Primer numero de la operacion</description>
        </parameter>
        <parameter>
            <name>segundo_operando</name>
            <type>int</type>
            <description>Segundo numero de la operacion</description>
        </parameter>
        <parameter>
            <name>operador</name>
            <type>str</type>
            <description>Operador aritmetico: +, -, *, /</description>
        </parameter>
    </parameters>
</tool_description>
"""

# Combinar ambas partes para formar el system prompt completo
system_prompt = system_tools_general + system_tools_specific

### Paso 2: Enviar un mensaje que requiera el uso de la herramienta

Se envia un mensaje del usuario cuya respuesta requiere el uso de la herramienta definida. El parametro `stop_sequences` es clave: hace que el modelo se detenga justo despues de generar la etiqueta de cierre `</function_calls>`, evitando que continue generando texto despues de la llamada.

In [ ]:
response = client.messages.create(
    model="claude-haiku-4-5-20251001",
    system=system_prompt,
    messages=[{
        "role": "user",
        "content": "Cuanto es 25 * 17?"
    }],
    max_tokens=500,
    stop_sequences=["</function_calls>"]
)

### Paso 3: El modelo genera la llamada a la herramienta

El modelo no responde directamente con el resultado. En lugar de eso, genera una solicitud estructurada indicando que herramienta quiere usar y con que parametros. Primero razona sobre lo que necesita hacer y luego genera la llamada en el formato XML definido.

La salida del modelo se vera algo asi:

```
Necesito calcular 25 * 17. Usare la calculadora.

<function_calls>
    <invoke name="calculadora">
        <parameter name="primer_operando">25</parameter>
        <parameter name="segundo_operando">17</parameter>
        <parameter name="operador">*</parameter>
    </invoke>
```

El codigo del desarrollador detecta la presencia de `</function_calls>` en el `stop_reason` y sabe que hay una llamada a herramienta pendiente de ejecutar.

### Paso 4: Ejecutar la funcion localmente

El desarrollador extrae los parametros de la respuesta del modelo y ejecuta la funcion real en su propio entorno. El modelo nunca ejecuta codigo; solo indica que funcion llamar y con que argumentos.

In [39]:
def calculadora(primer_operando, segundo_operando, operador):
    """Funcion real que ejecuta la operacion aritmetica."""
    operaciones = {
        "+": lambda a, b: a + b,
        "-": lambda a, b: a - b,
        "*": lambda a, b: a * b,
        "/": lambda a, b: a / b,
    }
    if operador not in operaciones:
        raise ValueError(f"Operador no soportado: {operador}")
    return operaciones[operador](primer_operando, segundo_operando)

# Extraer parametros de la respuesta del modelo y ejecutar
resultado = calculadora(25, 17, "*")
print(f"Resultado de la ejecucion local: {resultado}")

Resultado de la ejecucion local: 425


### Paso 5: Formatear el resultado

El resultado de la funcion debe empaquetarse en un formato XML especifico que el modelo reconoce. Notar que el texto comienza con `</function_calls>`: esto se debe a que el modelo se detuvo antes de generar esa etiqueta de cierre (por el `stop_sequences`), por lo que el desarrollador la agrega manualmente junto con el resultado.

In [ ]:
formatted_result = f"""
</function_calls>
<function_results>
<result>
<tool_name>calculadora</tool_name>
<stdout>
{resultado}
</stdout>
</result>
</function_results>
"""

print("Resultado formateado para devolver al modelo:")
print(formatted_result)

### Paso 6: Enviar el resultado de vuelta al modelo

Finalmente, se concatena la respuesta original del modelo con el resultado formateado y se envia todo de vuelta como continuacion de la conversacion. El modelo recibe el resultado de la herramienta y genera su respuesta final en lenguaje natural.

In [ ]:
# Concatenar la respuesta original del modelo + el resultado de la herramienta
full_response = response.content[0].text + formatted_result

messages = [
    {"role": "user", "content": "Cuanto es 25 * 17?"},
    {"role": "assistant", "content": full_response},
    {"role": "user", "content": "Presenta el resultado."}
]

final = client.messages.create(
    model="claude-haiku-4-5-20251001",
    system=system_prompt,
    messages=messages,
    max_tokens=500
)

print("Respuesta final del modelo:")
print(final.content[0].text)